In [ ]:
VER = 6  # 脚本版本

import pandas as pd, numpy as np  # 导入数据处理和数值计算库
from tqdm.notebook import tqdm  # Jupyter 环境下的进度条
tqdm.pandas()  # 为 pandas 操作启用进度条支持

import os, sys, pickle, glob, gc  # 导入常用标准库
from collections import Counter  # 导入计数工具
import cudf, itertools  # 导入 RAPIDS 和迭代工具
print('We will use RAPIDS version',cudf.__version__)  # 打印 RAPIDS 版本

pd.set_option('display.max_rows', 500)  # 设置最大显示行数
pd.set_option('display.max_columns', 500)  # 设置最大显示列数

from pandarallel import pandarallel  # pandas 并行处理工具

pandarallel.initialize(nb_workers=4, progress_bar=True, use_memory_fs=False)  # 初始化并行环境


# Candidate Generation

In [ ]:
GENERATE_FOR = "kaggle"  # 生成 kaggle 提交用的结果
CANDIDATE_COUNT = 100  # 每个会话生成的候选数量


In [ ]:
type_labels = {'clicks':0, 'carts':1, 'orders':2}  # 将行为类型映射为整数标签


In [ ]:
if GENERATE_FOR == "local":  # 如果生成本地验证集的候选集
    target_df = pd.read_parquet("./splitted_raw_data/val.parquet")  # 读取本地验证数据
    target_covisit = "local"  # 使用本地 co-visitation 文件
elif GENERATE_FOR == "kaggle":  # 如果生成 Kaggle 提交用的候选集
    target_df = pd.read_parquet("./splitted_raw_data/test.parquet")  # 读取测试数据
    target_covisit = "kaggle"  # 使用 Kaggle co-visitation 文件


In [ ]:
def pqt_to_dict(df):  # 把 parquet 表转换为 aid_x -> aid_y 列表字典
    return df.groupby('aid_x').aid_y.apply(list).to_dict()  # 按 aid_x 分组后转为字典

DISK_PIECES = 4  # 分硬盘片读取 co-visitation 结果

# 载入三类 co-visitation 矩阵
top_20_clicks = pqt_to_dict( pd.read_parquet(f'../raw_data/{target_covisit}_covisitation/{target_covisit}_top_{CANDIDATE_COUNT}_clicks_v{VER}_0.pqt') )  # 载入 clicks 矩阵的第 0 片

for k in range(1,DISK_PIECES):  # 依次载入 clicks 矩阵的其他片
    top_20_clicks.update( pqt_to_dict( pd.read_parquet(f'../raw_data/{target_covisit}_covisitation/{target_covisit}_top_{CANDIDATE_COUNT}_clicks_v{VER}_{k}.pqt') ) )  # 合并 clicks 矩阵


top_20_buys = pqt_to_dict( pd.read_parquet(f'../raw_data/{target_covisit}_covisitation/{target_covisit}_top_{CANDIDATE_COUNT}_carts_orders_v{VER}_0.pqt') )  # 载入 carts+orders 矩阵的第 0 片

for k in range(1,DISK_PIECES):  # 依次载入 carts+orders 矩阵的其他片
    top_20_buys.update( pqt_to_dict( pd.read_parquet(f'../raw_data/{target_covisit}_covisitation/{target_covisit}_top_{CANDIDATE_COUNT}_carts_orders_v{VER}_{k}.pqt') ) )  # 合并 carts+orders 矩阵

top_20_buy2buy = pqt_to_dict( pd.read_parquet(f'../raw_data/{target_covisit}_covisitation/{target_covisit}_top_{CANDIDATE_COUNT}_buy2buy_v{VER}_0.pqt') )  # 载入 buy2buy 矩阵

print('Here are size of our 3 co-visitation matrices:')  # 打印三个 co-visitation 矩阵的大小
print( len( top_20_clicks ), len( top_20_buy2buy ), len( top_20_buys ) )  # 输出各矩阵的条目数


In [ ]:
top_clicks = target_df.loc[target_df['type']== 0,'aid'].value_counts().index.values[:CANDIDATE_COUNT]  # 获取 click 最常见的 aid
top_carts = target_df.loc[target_df['type']== 1,'aid'].value_counts().index.values[:CANDIDATE_COUNT]  # 获取 cart 最常见的 aid
top_orders = target_df.loc[target_df['type']== 2,'aid'].value_counts().index.values[:CANDIDATE_COUNT]  # 获取 order 最常见的 aid


In [ ]:
type_weight_multipliers = {0: 1, 1: 5, 2: 4}  # 不同行为类型的权重系数

def suggest_clicks(df):  # 生成 clicks 任务的候选集
    # 用户历史 aid 和 type
    aids=df.aid.tolist()  # 提取 aid 列表
    types = df.type.tolist()  # 提取 type 列表
    unique_aids = list(dict.fromkeys(aids[::-1] ))  # 按时间倒序保留去重 aid
    # 使用权重重排候选集
    if len(unique_aids)>=CANDIDATE_COUNT:  # 如果历史 aid 已经足够多
        weights=np.logspace(0.1,1,len(aids),base=2, endpoint=True)-1  # 根据位置生成权重
        aids_temp = Counter()  # 用 Counter 统计位置权重
        # 基于重复物品和行为类型重排
        for aid,w,t in zip(aids,weights,types):  # 依次遍历 aid、权重和 type
            aids_temp[aid] += w * type_weight_multipliers[t]  # 累加每个 aid 的分数
        sorted_aids = [k for k,v in aids_temp.most_common(CANDIDATE_COUNT)]  # 按分数取前 CANDIDATE_COUNT 个
        return sorted_aids  # 直接返回
    # 使用 clicks co-visitation 矩阵
    aids2 = list(itertools.chain(*[top_20_clicks[aid] for aid in unique_aids if aid in top_20_clicks]))  # 拼接相关 aid
    # 重排候选集
    top_aids2 = [aid2 for aid2, cnt in Counter(aids2).most_common(CANDIDATE_COUNT) if aid2 not in unique_aids]  # 去掉已出现 aid
    result = unique_aids + top_aids2[:CANDIDATE_COUNT - len(unique_aids)]  # 先用历史 aid 再补等长
    # 用 clicks 整体热门 aid 补充
    return result + list(top_clicks)[:CANDIDATE_COUNT-len(result)]  # 不够时用全局 click 热门补全


In [ ]:
def suggest_carts(df):  # 生成 carts 任务的候选集
    # 用户历史 aid 和 type
    aids = df.aid.tolist()  # 提取 aid 列表
    types = df.type.tolist()  # 提取 type 列表

    # 去重 aid 和去重 buy
    unique_aids = list(dict.fromkeys(aids[::-1] ))  # 按时间倒序保留去重 aid
    df = df.loc[(df['type'] == 0)|(df['type'] == 1)]  # 只保留 click 和 cart 记录
    unique_buys = list(dict.fromkeys(df.aid.tolist()[::-1]))  # 按时间倒序保留去重 buy

    # 使用权重重排候选集
    if len(unique_aids) >= CANDIDATE_COUNT:  # 如果历史 aid 足够多
        weights=np.logspace(0.5,1,len(aids),base=2, endpoint=True)-1  # 根据位置生成权重
        aids_temp = Counter()  # 用 Counter 累加分数

        # 基于重复物品和行为类型重排
        for aid,w,t in zip(aids,weights,types):  # 依次遍历 aid、权重和 type
            aids_temp[aid] += w * type_weight_multipliers[t]  # 累加每个 aid 的分数

        # 用 carts+orders co-visitation 矩阵进一步补充
        aids2 = list(itertools.chain(*[top_20_buys[aid] for aid in unique_buys if aid in top_20_buys]))  # 拼接相关 aid
        for aid in aids2: aids_temp[aid] += 0.1  # 给 co-visitation 命中的 aid 加一点额外分
        sorted_aids = [k for k,v in aids_temp.most_common(CANDIDATE_COUNT)]  # 按分数取前 CANDIDATE_COUNT 个
        return sorted_aids  # 直接返回

    # 使用 carts 和 clicks co-visitation 矩阵
    aids1 = list(itertools.chain(*[top_20_clicks[aid] for aid in unique_aids if aid in top_20_clicks]))  # 拼接 clicks 相关 aid
    aids2 = list(itertools.chain(*[top_20_buys[aid] for aid in unique_aids if aid in top_20_buys]))  # 拼接 carts/orders 相关 aid

    # 重排候选集
    top_aids2 = [aid2 for aid2, cnt in Counter(aids1+aids2).most_common(CANDIDATE_COUNT) if aid2 not in unique_aids]  # 去掉已出现 aid
    result = unique_aids + top_aids2[:CANDIDATE_COUNT - len(unique_aids)]  # 先用历史 aid 再补等长

    # 用 carts 热门 aid 补充
    return result + list(top_carts)[:CANDIDATE_COUNT-len(result)]  # 不够时用全局 cart 热门补全


In [ ]:
def suggest_buys(df):  # 生成 buys/orders 任务的候选集
    # 用户历史 aid 和 type
    aids=df.aid.tolist()  # 提取 aid 列表
    types = df.type.tolist()  # 提取 type 列表
    # 去重 aid 和去重 buy
    unique_aids = list(dict.fromkeys(aids[::-1] ))  # 按时间倒序保留去重 aid
    df = df.loc[(df['type']==1)|(df['type']==2)]  # 只保留 cart 和 order 记录
    unique_buys = list(dict.fromkeys( df.aid.tolist()[::-1] ))  # 按时间倒序保留去重 buy
    # 使用权重重排候选集
    if len(unique_aids)>=CANDIDATE_COUNT:  # 如果历史 aid 足够多
        weights=np.logspace(0.5,1,len(aids),base=2, endpoint=True)-1  # 根据位置生成权重
        aids_temp = Counter()  # 用 Counter 累加分数
        # 基于重复物品和行为类型重排
        for aid,w,t in zip(aids,weights,types):  # 依次遍历 aid、权重和 type
            aids_temp[aid] += w * type_weight_multipliers[t]  # 累加每个 aid 的分数
        # 用 buy2buy co-visitation 矩阵进一步补充
        aids3 = list(itertools.chain(*[top_20_buy2buy[aid] for aid in unique_buys if aid in top_20_buy2buy]))  # 拼接相关 aid
        for aid in aids3: aids_temp[aid] += 0.1  # 给 co-visitation 命中的 aid 加一点额外分
        sorted_aids = [k for k,v in aids_temp.most_common(CANDIDATE_COUNT)]  # 按分数取前 CANDIDATE_COUNT 个
        return sorted_aids  # 直接返回
    # 使用 carts/order co-visitation 矩阵
    aids2 = list(itertools.chain(*[top_20_buys[aid] for aid in unique_aids if aid in top_20_buys]))  # 拼接 carts/orders 相关 aid
    # 使用 buy2buy co-visitation 矩阵
    aids3 = list(itertools.chain(*[top_20_buy2buy[aid] for aid in unique_buys if aid in top_20_buy2buy]))  # 拼接 buy2buy 相关 aid
    # 重排候选集
    top_aids2 = [aid2 for aid2, cnt in Counter(aids2+aids3).most_common(CANDIDATE_COUNT) if aid2 not in unique_aids]  # 去掉已出现 aid
    result = unique_aids + top_aids2[:CANDIDATE_COUNT - len(unique_aids)]  # 先用历史 aid 再补等长
    # 用 orders 热门 aid 补充
    return result + list(top_orders)[:CANDIDATE_COUNT-len(result)]  # 不够时用全局 order 热门补全


# Create Submission CSV
Inferring test data with Pandas groupby is slow. We need to accelerate the following code.

In [ ]:
pred_df_clicks = target_df.sort_values(["session", "ts"]).groupby(["session"]).parallel_apply(  # 按 session 和 ts 排序后并行生成 clicks 候选
    lambda x: suggest_clicks(x)  # 对每个 session 调用 clicks 生成函数
)  # 收集 clicks 结果

pred_df_carts = target_df.sort_values(["session", "ts"]).groupby(["session"]).parallel_apply(  # 按 session 和 ts 排序后并行生成 carts 候选
    lambda x: suggest_carts(x)  # 对每个 session 调用 carts 生成函数
)  # 收集 carts 结果

pred_df_buys = target_df.sort_values(["session", "ts"]).groupby(["session"]).parallel_apply(  # 按 session 和 ts 排序后并行生成 orders 候选
    lambda x: suggest_buys(x)  # 对每个 session 调用 orders 生成函数
)  # 收集 orders 结果


In [ ]:
clicks_pred_df = pd.DataFrame(pred_df_clicks, columns=["aid"]).reset_index()  # 将 clicks 结果转为 DataFrame
orders_pred_df = pd.DataFrame(pred_df_buys, columns=["aid"]).reset_index()  # 将 orders 结果转为 DataFrame
carts_pred_df = pd.DataFrame(pred_df_carts, columns=["aid"]).reset_index()  # 将 carts 结果转为 DataFrame


In [ ]:
clicks_pred_df.explode(column="aid").to_parquet(f"./candidate_data/{target_covisit}_{CANDIDATE_COUNT}candidates_clicks.parquet")  # 展开 clicks 候选并保存
orders_pred_df.explode(column="aid").to_parquet(f"./candidate_data/{target_covisit}_{CANDIDATE_COUNT}candidates_orders.parquet")  # 展开 orders 候选并保存
carts_pred_df.explode(column="aid").to_parquet(f"./candidate_data/{target_covisit}_{CANDIDATE_COUNT}candidates_carts.parquet")  # 展开 carts 候选并保存
